# Resnet-18 for WT and NWT for unique permutations ( Picked 5 strains per superclass for balanced results)  - fix scale issue

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torchvision.models as models
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import random
import itertools
import math

def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

In [ ]:
# Define strain mappings for WT vs NWT
WT_STRAINS = [0, 1, 2, 3, 4]      # 5 strains, class 0
NWT_STRAINS = [17, 12, 5, 9, 8]   # 5 strains, class 1
ORDER = WT_STRAINS + NWT_STRAINS  # total_strains = 10

print(f"WT strains: {WT_STRAINS}")
print(f"NWT strains: {NWT_STRAINS}")
print(f"Total order: {ORDER}")

In [ ]:
# Cell 5: Robust data loading function (adapted from KP_ABV1)
import numpy as np
import scipy.io
from scipy.io.matlab.mio5_params import mat_struct

WT_STRAINS = [0, 1, 2, 3, 4]
NWT_STRAINS = [17, 12, 5, 9, 8]
TARGET_ORDER = WT_STRAINS + NWT_STRAINS

# converters and finders
def _deep(o):
    if isinstance(o, mat_struct):
        return {f: _deep(getattr(o, f)) for f in o._fieldnames}
    if isinstance(o, np.ndarray):
        if o.dtype == object:
            if o.ndim == 0: return _deep(o.item())
            return [_deep(x) for x in o.ravel()]
        if o.dtype.names is not None:
            if o.shape == ():  # scalar structured
                return {n: _deep(o[n]) for n in o.dtype.names}
            return [{n: _deep(o[idx][n]) for n in o.dtype.names}
                    for idx in np.ndindex(o.shape)]
        if o.ndim == 0: return o.item()
        return o
    if isinstance(o, np.void) and o.dtype.names is not None:
        return {n: _deep(o[n]) for n in o.dtype.names}
    return o

def _load_var_v5(mat_path, varname):
    part = scipy.io.loadmat(
        mat_path,
        variable_names=[varname],
        squeeze_me=False,
        struct_as_record=True,
        chars_as_strings=True,
    )
    if varname not in part:
        raise KeyError(f"{varname} not found in {mat_path}")
    return _deep(part[varname])

def _coerce_60x60(x):
    a = np.asarray(x, dtype=np.float32)
    a = np.squeeze(a)
    if a.shape == (60, 60):
        return a
    if a.ndim == 3:
        for ax in range(3):
            if a.shape[ax] == 1:
                b = np.squeeze(a, axis=ax)
                if b.shape == (60, 60):
                    return b.astype(np.float32)
    raise ValueError(f"Unexpected image shape {a.shape}")

def _find_image(o):
    if isinstance(o, dict):
        if "image" in o: return _coerce_60x60(o["image"])
        for v in o.values():
            try: return _find_image(v)
            except Exception: pass
        raise KeyError("No 'image' in dict")
    if isinstance(o, (list, tuple)):
        for v in o:
            try: return _find_image(v)
            except Exception: pass
        raise KeyError("No 'image' in list/tuple")
    if isinstance(o, np.ndarray):
        if o.dtype == object:
            for v in o.ravel():
                try: return _find_image(v)
                except Exception: pass
            raise KeyError("No 'image' in object array")
        if o.dtype.names is not None:
            for n in o.dtype.names:
                try: return _find_image(o[n])
                except Exception: pass
            raise KeyError("No 'image' in structured array")
        return _coerce_60x60(o)
    if isinstance(o, np.void) and o.dtype.names is not None:
        for n in o.dtype.names:
            try: return _find_image(o[n])
            except Exception: pass
        raise KeyError("No 'image' in np.void")
    raise TypeError(f"Unsupported item type {type(o)}")

def f_read_bacteria_WT_NWT_py(mat_path: str, n_cells_perS: int):
    usable_order = []
    loaded_cells_per_strain = {}

    # Try to load each target strain individually
    for s in TARGET_ORDER:
        name = f'class_{s}'
        try:
            cells = _load_var_v5(mat_path, name)
            # normalize to list
            if isinstance(cells, np.ndarray) and cells.dtype == object:
                cells = [_deep(x) for x in cells.ravel()]
            elif not isinstance(cells, list):
                cells = [cells]
            if len(cells) >= n_cells_perS:
                usable_order.append(s)
                loaded_cells_per_strain[s] = cells
            else:
                raise ValueError(f"{name} has only {len(cells)} items (< {n_cells_perS})")
        except Exception as e:
            print(f"✖ Skipping {name}: {e}")
            continue

    if not usable_order:
        raise RuntimeError("No expected class_* variables could be loaded from this file.")

    S = len(usable_order)
    H = W = 60
    N = S * n_cells_perS
    X = np.empty((H, W, N), dtype=np.float32)
    strain_id = np.empty((N, 1), dtype=np.int32)
    class_id  = np.empty((N, 1), dtype=np.int32)

    ptr = 0
    for j, strain in enumerate(usable_order, start=1):
        cells = loaded_cells_per_strain[strain]
        # take first n_cells_perS
        for i in range(n_cells_perS):
            X[:, :, ptr + i] = _find_image(cells[i])
        strain_id[ptr:ptr+n_cells_perS, 0] = j
        class_id[ptr:ptr+n_cells_perS, 0]  = (0 if strain in WT_STRAINS else 1)
        ptr += n_cells_perS

    strain_ids_0 = np.asarray(usable_order, dtype=np.int32)
    strain_ids   = np.arange(1, S + 1, dtype=np.int32)
    return X, strain_ids_0, strain_ids, strain_id, class_id

In [ ]:
# Load data (adjust paths as needed)
#TRAIN
X_tr, s0_tr, s_tr, sid_tr, cid_tr = f_read_bacteria_WT_NWT_py(
    '/home/dineth/bacteria_QPM/data/bacteria_train_full_data.mat',
    n_cells_perS=11000
)

# TEST
X_te, s0_te, s_te, sid_te, cid_te = f_read_bacteria_WT_NWT_py(
    '/home/dineth/bacteria_QPM/data/bacteria_test_full_data.mat',
    n_cells_perS=1300
)

In [ ]:
# Print data information
print("\n TRAIN")
print("X_tr shape:", X_tr.shape)
print("strain_ids_0:", s0_tr)
print("strain_ids:", s_tr)
print("strain_id shape:", sid_tr.shape, "| unique:", np.unique(sid_tr))
print("class_id shape:", cid_tr.shape, "| unique:", np.unique(cid_tr))
print("Counts per class:")
for c in np.unique(cid_tr):
    count = np.sum(cid_tr == c)
    print(f"  Class {c}: {count} samples")

print("\n TEST ")
print("X_te shape:", X_te.shape)
print("strain_ids_0:", s0_te)
print("strain_ids:", s_te)
print("strain_id shape:", sid_te.shape, "| unique:", np.unique(sid_te))
print("class_id shape:", cid_te.shape, "| unique:", np.unique(cid_te))

In [ ]:
# ResNet-18 Binary Classifier
class ResNet18BinaryClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super(ResNet18BinaryClassifier, self).__init__()
        self.resnet = models.resnet18(pretrained=False)
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)
        
    def forward(self, x):
        return self.resnet(x)

In [ ]:
# Data preparation function
import torchvision.transforms as transforms

def prepare_data(X, y, batch_size=32, shuffle=True):
    """Convert numpy arrays to PyTorch DataLoader"""
    # Reshape X from (60, 60, N) to (N, 1, 60, 60) for PyTorch
    X_reshaped = np.transpose(X, (2, 0, 1))  # (N, 60, 60)
    X_reshaped = np.expand_dims(X_reshaped, axis=1)  # (N, 1, 60, 60)
    
    X_tensor = torch.FloatTensor(X_reshaped)
    y_tensor = torch.LongTensor(y.flatten())
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        # Calculate accuracy
        bs = target.size(0)
        total_loss += loss.detach().item() * bs
        pred = output.detach().argmax(dim=1)
        correct += (pred == target).sum().item()
        total += bs

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc

In [ ]:
# Evaluation function
def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            bs = target.size(0)
            total_loss += loss.detach().item() * bs
            pred = output.detach().argmax(dim=1)

            correct += (pred == target).sum().item()
            total += bs
            all_preds.extend(pred.cpu().tolist())
            all_targets.extend(target.cpu().tolist())

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc, all_preds, all_targets

In [ ]:
def train_eval_permutation_with_epochs(
    X_train, y_train, X_test, y_test, epochs=8, lr=0.01, batch_size=64
):
    """Train ResNet-18 with SGD optimizer for permutation test"""
    
    torch.manual_seed(123)
    torch.cuda.manual_seed_all(123)
    
    # Create new model for each permutation
    model = ResNet18BinaryClassifier().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    train_loader = prepare_data(X_train, y_train, batch_size=batch_size, shuffle=True)
    test_loader = prepare_data(X_test, y_test, batch_size=batch_size, shuffle=False)

    best_test_acc = -1.0
    best_epoch = -1
    best_state = None

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        
        # Evaluate on test set
        model.eval()
        with torch.no_grad():
            test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, DEVICE)

        print(f"    Epoch {epoch}: Train Acc = {train_acc:.2f}%, Test Acc = {test_acc:.2f}%")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"[BEST] Epoch {best_epoch}: Test Acc = {best_test_acc:.2f}%")
    return best_test_acc / 100.0  # here make return best_test_acc

# Cell 13: Permutation test parameters
N_PERMUTATIONS = 252
PERM_EPOCHS = 8  # 8 epochs to match Frontiers paper
RANDOM_SEED = 42


In [ ]:
# Permutation test setup
# Calculate the exact number of unique permutations
total_strains = 10
wt_strains = 5  # number of 0s
nwt_strains = 5  # number of 1s

unique_permutations_count = math.factorial(total_strains) // (math.factorial(wt_strains) * math.factorial(nwt_strains))
print(f"Number of unique permutations: {unique_permutations_count}")

# Generate all unique permutations
def generate_all_unique_permutations():
    # Create the base pattern: 5 zeros and 5 ones
    base_pattern = [0] * wt_strains + [1] * nwt_strains
    
    # Generate all unique permutations of this pattern
    unique_patterns = list(set(itertools.permutations(base_pattern)))
    
    print(f"Generated {len(unique_patterns)} unique patterns")
    return unique_patterns

# Generate all unique permutation patterns
all_unique_patterns = generate_all_unique_permutations()

# Permutation test parameters
N_PERMUTATIONS = len(all_unique_patterns)  # This will be 252
PERM_EPOCHS = 8
RANDOM_SEED = 42

print(f"\nRunning {N_PERMUTATIONS} unique permutations...")

In [ ]:
# True Labels Training
print(f"\nTRUE labels training:")
true_accuracy = train_eval_permutation_with_epochs(
    X_tr, cid_tr,  # Using train data
    X_te, cid_te,  # Using test data
    epochs=PERM_EPOCHS
)
print(f"TRUE labels final accuracy: {true_accuracy:.2%}")

In [ ]:
# Permutation test execution
# Get strain IDs for each sample - use original strain numbers, not mapped IDs
# Create mapping from mapped strain IDs (1-10) back to original strain numbers
import time

# Start timing
start_time = time.time()

id_to_original = {mapped_id: original_id for mapped_id, original_id in zip(s_tr, s0_tr)}

# Get strain IDs for each sample - convert mapped IDs back to original strain numbers
ref_strains_order = np.array([id_to_original[sid] for sid in sid_tr.flatten()])
te_strains_order = np.array([id_to_original[sid] for sid in sid_te.flatten()])

null_accuracies = []
LAST_PERM_MAP = None

# Define the true pattern to track
true_pattern = tuple([0] * wt_strains + [1] * nwt_strains)  # (0,0,0,0,0,1,1,1,1,1)
true_perm_idx = None
true_accuracy = None  # Will be extracted from permutations

def print_permutation_mapping(perm_idx, y_ref_perm, y_te_perm):
    print(f"\nPermutation {perm_idx}: strain -> class (TRUE -> PERMUTED)")
    if LAST_PERM_MAP is None:
        print("  [warn] LAST_PERM_MAP is not set")
        return
    for sid in ORDER:
        true_class = 0 if sid in WT_STRAINS else 1
        perm_class = LAST_PERM_MAP[sid]
        print(f"Strain {sid:2d} | {'WT' if true_class == 0 else 'NWT':<3} -> {'WT' if perm_class == 0 else 'NWT'}")

for perm_idx, perm_pattern in enumerate(all_unique_patterns, 1):
    # Check if this is the true pattern
    is_true_pattern = (tuple(perm_pattern) == true_pattern)
    if is_true_pattern:
        true_perm_idx = perm_idx
        print(f"\n{'='*60}")
        print(f"*** THIS IS THE TRUE PATTERN (Permutation {perm_idx}) ***")
        print(f"{'='*60}")
    
    # Create strain-to-label mapping using the current unique pattern
    strain_to_label = {strain: perm_pattern[i] for i, strain in enumerate(ORDER)}
    
    # Apply the mapping to both splits
    y_ref_permuted = np.array([strain_to_label[s] for s in ref_strains_order], dtype=int)
    y_te_permuted = np.array([strain_to_label[s] for s in te_strains_order], dtype=int)
    
    LAST_PERM_MAP = strain_to_label
    print_permutation_mapping(perm_idx, y_ref_permuted, y_te_permuted)
    
    # Train & evaluate with permuted labels
    print(f"\nPermutation {perm_idx} training:")
    perm_accuracy = train_eval_permutation_with_epochs(
        X_tr, y_ref_permuted,
        X_te, y_te_permuted,
        epochs=PERM_EPOCHS
    )
    null_accuracies.append(perm_accuracy)
    
    # Store the true accuracy when found
    if is_true_pattern:
        true_accuracy = perm_accuracy
        print(f"\n{'='*60}")
        print(f"*** TRUE ACCURACY EXTRACTED: {true_accuracy:.4f} ({true_accuracy*100:.2f}%) ***")
        print(f"*** From Permutation {perm_idx} ***")
        print(f"{'='*60}")
    
    if perm_idx % 20 == 0:  # Progress update every 20 permutations
        print(f"Completed {perm_idx}/{N_PERMUTATIONS} permutations")

print("Permutation test completed!")

# Verify true accuracy was found
if true_accuracy is None:
    print("WARNING: True pattern not found in permutations!")
else:
    print(f"\n{'='*60}")
    print(f"SUMMARY:")
    print(f"True pattern found at Permutation {true_perm_idx}")
    print(f"True accuracy (from permutations): {true_accuracy:.4f} ({true_accuracy*100:.2f}%)")
    print(f"{'='*60}")

# End timing
end_time = time.time()
total_time = end_time - start_time
avg_perm_time = total_time / N_PERMUTATIONS

print(f"\n[CELL 10 TIMING]")
print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"Average time per permutation: {avg_perm_time:.2f} seconds")
print(f"Estimated time for 1000 permutations: {avg_perm_time * 1000 / 60:.2f} minutes")